# 03 Model training

* Baselines: constant training base rate, and the smoothed historical route delay rate.
* Logistic regression (L2) and XGBoost, hyperparameters picked on **validation PR-AUC** (Oct-Nov).
* Calibration: Platt vs isotonic picked on validation; plus a rolling Platt recalibration for the drift into December.
* Feature-group ablation.

December (test) is only scored here, never used for any choice.

In [1]:
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

In [2]:
from pathlib import Path

ROOT = Path("..").resolve()
RAW_CSV = ROOT / "data_raw" / "flights.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
FIG_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
for d in (PROCESSED_DIR, MODEL_DIR, FIG_DIR, TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42

In [3]:
with open(PROCESSED_DIR / "feature_columns.json") as f:
    cols = json.load(f)
CAT_COLS, NUM_COLS = cols["cat"], cols["num"]
ALL_FEATURES = CAT_COLS + NUM_COLS


def load_splits():
    df = pd.read_pickle(PROCESSED_DIR / "features.pkl")
    df = df[~df["no_arrival"]]
    return {s: df[df["split"] == s].reset_index(drop=True) for s in ("train", "val", "test")}


splits = load_splits()
train, val, test = splits["train"], splits["val"], splits["test"]
categories = {c: sorted(train[c].unique()) for c in CAT_COLS}
xgb_cols = CAT_COLS + NUM_COLS
{k: len(v) for k, v in splits.items()}

{'train': 244737, 'val': 55589, 'test': 27020}

## Model definitions

In [4]:
def make_lr(num_cols, cat_cols, C):
    pre = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), cat_cols),
    ])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(C=C, max_iter=5000))])

def to_xgb_frame(df, cols, categories):
    X = df[cols].copy()
    for c in CAT_COLS:
        if c in X:
            # levels never seen in training (a few new destinations) become missing
            known = X[c].where(X[c].isin(categories[c]))
            X[c] = pd.Categorical(known, categories=categories[c])
    return X


def make_xgb(params):
    return XGBClassifier(
        n_estimators=3000, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        tree_method="hist", enable_categorical=True, max_cat_to_onehot=1,
        eval_metric="aucpr", early_stopping_rounds=100,
        random_state=SEED, n_jobs=-1, **params,
    )


def fit_xgb(train, val, cols, categories, params):
    model = make_xgb(params)
    model.fit(to_xgb_frame(train, cols, categories), train["delayed"],
              eval_set=[(to_xgb_frame(val, cols, categories), val["delayed"])], verbose=False)
    return model

## Logistic regression

In [5]:
LR_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]

lr_rows = []
for C in LR_GRID:
    model = make_lr(NUM_COLS, CAT_COLS, C).fit(train, train["delayed"])
    ap = average_precision_score(val["delayed"], model.predict_proba(val)[:, 1])
    lr_rows.append({"model": "lr", "C": C, "val_pr_auc": ap})
    print(f"C={C:<6} val PR-AUC={ap:.4f}")

lr_best = max(lr_rows, key=lambda r: r["val_pr_auc"])
lr = make_lr(NUM_COLS, CAT_COLS, lr_best["C"]).fit(train, train["delayed"])
lr_best

C=0.001  val PR-AUC=0.2941


C=0.01   val PR-AUC=0.2959


C=0.1    val PR-AUC=0.2972


C=1.0    val PR-AUC=0.2974


C=10.0   val PR-AUC=0.2975


{'model': 'lr', 'C': 10.0, 'val_pr_auc': 0.29746886732066646}

## XGBoost

Large `min_child_weight` values on purpose. Several features are constant within a day (yesterday's rates), so together
they identify the date; with small leaves the trees memorise individual training days. Since the hessian
$h = p(1-p) \le 1/4$, `min_child_weight=2000` means at least 8,000 flights per leaf.

In [6]:
XGB_GRID = [
    {"max_depth": d, "min_child_weight": w, "reg_lambda": lam}
    for d in (3, 4, 6) for w in (100, 500, 2000) for lam in (1, 20)
]

xgb_rows, xgb, best_ap = [], None, -1
for params in XGB_GRID:
    model = fit_xgb(train, val, xgb_cols, categories, params)
    ap = average_precision_score(val["delayed"], model.predict_proba(to_xgb_frame(val, xgb_cols, categories))[:, 1])
    xgb_rows.append({"model": "xgb", **params, "n_trees": model.best_iteration + 1, "val_pr_auc": ap})
    print(f"{params} trees={model.best_iteration + 1} val PR-AUC={ap:.4f}")
    if ap > best_ap:
        xgb, best_ap = model, ap

xgb_best = max(xgb_rows, key=lambda r: r["val_pr_auc"])
xgb_best

OMP: Warning #179: Function Can't set size of /tmp file failed:


{'max_depth': 3, 'min_child_weight': 100, 'reg_lambda': 1} trees=156 val PR-AUC=0.2701


{'max_depth': 3, 'min_child_weight': 100, 'reg_lambda': 20} trees=199 val PR-AUC=0.2714


{'max_depth': 3, 'min_child_weight': 500, 'reg_lambda': 1} trees=203 val PR-AUC=0.2713


{'max_depth': 3, 'min_child_weight': 500, 'reg_lambda': 20} trees=178 val PR-AUC=0.2711


...

{'max_depth': 6, 'min_child_weight': 500, 'reg_lambda': 1} trees=69 val PR-AUC=0.2703


{'max_depth': 6, 'min_child_weight': 500, 'reg_lambda': 20} trees=93 val PR-AUC=0.2709


{'max_depth': 6, 'min_child_weight': 2000, 'reg_lambda': 1} trees=235 val PR-AUC=0.2750


{'max_depth': 6, 'min_child_weight': 2000, 'reg_lambda': 20} trees=94 val PR-AUC=0.2738


{'model': 'xgb',
 'max_depth': 4,
 'min_child_weight': 2000,
 'reg_lambda': 20,
 'n_trees': 308,
 'val_pr_auc': 0.275541873936498}

In [7]:
raw = {}
for name, df in (("val", val), ("test", test)):
    raw[name] = {
        "lr": lr.predict_proba(df)[:, 1],
        "xgb": xgb.predict_proba(to_xgb_frame(df, xgb_cols, categories))[:, 1],
    }

# in-sample scores, only to show the train/validation gap
train_fit = {}
for m, p in (("lr", lr.predict_proba(train)[:, 1]),
             ("xgb", xgb.predict_proba(to_xgb_frame(train, xgb_cols, categories))[:, 1])):
    train_fit[m] = {"pr_auc": average_precision_score(train["delayed"], p),
                    "roc_auc": roc_auc_score(train["delayed"], p)}
pd.DataFrame(train_fit).round(4)

,lr,xgb
pr_auc,0.4524,0.5256
roc_auc,0.7101,0.7520


## Calibration

* **Platt scaling:** logistic regression on the logit of the score.
* **Isotonic regression:** monotone step function.

Choice: fit on October and score the Brier score on November, and the other way round; the winner is refitted on
all of validation.

In [8]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


class Calibrator:
    """Platt scaling (logistic on the logit of the score) or isotonic regression."""

    def __init__(self, method):
        self.method = method

    def fit(self, p, y):
        if self.method == "sigmoid":
            self.model = LogisticRegression(C=1e6).fit(logit(p)[:, None], y)
        else:
            self.model = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y)
        return self

    def predict(self, p):
        if self.method == "sigmoid":
            return self.model.predict_proba(logit(p)[:, None])[:, 1]
        return self.model.predict(p)


def choose_calibrator(p, y, month):
    scores = {}
    for method in ("sigmoid", "isotonic"):
        briers = []
        for fit_m, eval_m in ((10, 11), (11, 10)):
            cal = Calibrator(method).fit(p[month == fit_m], y[month == fit_m])
            briers.append(brier_score_loss(y[month == eval_m], cal.predict(p[month == eval_m])))
        scores[method] = float(np.mean(briers))
    best = min(scores, key=scores.get)
    return Calibrator(best).fit(p, y), scores


calibrators, cal_scores = {}, {}
for m in ("lr", "xgb"):
    calibrators[m], cal_scores[m] = choose_calibrator(
        raw["val"][m], val["delayed"].to_numpy(), val["month"].to_numpy())
    print(f"{m}: cross-month Brier {cal_scores[m]} -> {calibrators[m].method}")

lr: cross-month Brier {'sigmoid': 0.13894430653670106, 'isotonic': 0.13907106307741604} -> sigmoid
xgb: cross-month Brier {'sigmoid': 0.14066383242607117, 'isotonic': 0.1406693384051323} -> sigmoid


### Rolling recalibration

October-November had about 18% delayed flights, December about 34%. A calibrator fitted on Oct-Nov therefore
under-predicts December. The rolling version refits Platt scaling every day on the previous `w` days of scores and
outcomes (all known at the decision time). Only out-of-sample (val/test) scores are used as history.
The window is chosen on November (history = Oct-Nov), then December is recalibrated day by day.

In [9]:
ROLL_WINDOWS = (7, 14, 28)


def rolling_calibrate(history, target, col, window):
    out = target[col].to_numpy().copy()   # days without usable history keep the raw score
    for d in target["date"].unique():
        past = history[(history["date"] < d) & (history["date"] >= d - pd.Timedelta(days=window))]
        if past["delayed"].nunique() < 2:
            continue
        cal = Calibrator("sigmoid").fit(past[col].to_numpy(), past["delayed"].to_numpy())
        idx = (target["date"] == d).to_numpy()
        out[idx] = cal.predict(out[idx])
    return out


train_rate = train["delayed"].mean()
keep = ["id", "date", "month", "carrier", "origin", "dest", "hour", "delayed"]
preds = {}
for name, df in (("val", val), ("test", test)):
    out = df[keep].copy()
    out["p_prior"] = train_rate
    out["p_hist_route"] = df["hist_rate_route"].fillna(train_rate)
    for m in ("lr", "xgb"):
        out[f"p_{m}_raw"] = raw[name][m]
        out[f"p_{m}_cal"] = calibrators[m].predict(raw[name][m])
    preds[name] = out

history = pd.concat([preds["val"], preds["test"]], ignore_index=True)
roll_scores = {}
nov = (preds["val"]["month"] == 11).to_numpy()
for m in ("lr", "xgb"):
    briers = {}
    for w in ROLL_WINDOWS:
        p = rolling_calibrate(preds["val"], preds["val"], f"p_{m}_raw", w)
        briers[w] = brier_score_loss(preds["val"]["delayed"][nov], p[nov])
    w_best = min(briers, key=briers.get)
    print(f"{m}: November Brier by window {briers} -> {w_best} days")
    preds["val"][f"p_{m}_roll"] = rolling_calibrate(preds["val"], preds["val"], f"p_{m}_raw", w_best)
    preds["test"][f"p_{m}_roll"] = rolling_calibrate(history, preds["test"], f"p_{m}_raw", w_best)
    roll_scores[m] = {"nov_brier": briers, "window": w_best}

lr: November Brier by window {7: 0.14288307110289977, 14: 0.1434612945733144, 28: 0.1420487166006755} -> 28 days


xgb: November Brier by window {7: 0.1435622125864029, 14: 0.14396634697914124, 28: 0.14310410618782043} -> 28 days


## Save predictions, models and the selected settings

In [10]:
for name, out in preds.items():
    out.to_pickle(PROCESSED_DIR / f"predictions_{name}.pkl")

joblib.dump(lr, MODEL_DIR / "lr.joblib")
xgb.save_model(str(MODEL_DIR / "xgb.json"))
with open(MODEL_DIR / "categories.json", "w") as f:
    json.dump(categories, f)

pd.DataFrame(lr_rows + xgb_rows).to_csv(RESULTS_DIR / "tuning.csv", index=False)
with open(RESULTS_DIR / "selected_models.json", "w") as f:
    json.dump({
        "lr": lr_best, "xgb": xgb_best,
        "train_fit": train_fit,
        "calibration_cv_brier": cal_scores,
        "calibration_method": {m: c.method for m, c in calibrators.items()},
        "rolling_calibration": roll_scores,
        "train_positive_rate": float(train_rate),
        "n_rows": {k: len(v) for k, v in splits.items()},
    }, f, indent=2)
print("saved")

saved


## Feature-group ablation

Add one feature group at a time and refit both models with the hyperparameters chosen above.

In [11]:
groups = cols["groups"]
GROUPS = [
    ("schedule + calendar", groups["calendar"] + groups["schedule"]),
    ("+ congestion", groups["congestion"]),
    ("+ long-run history", groups["long_history"]),
    ("+ recent history", groups["recent_history"]),
]
xgb_params = {k: xgb_best[k] for k in ("max_depth", "min_child_weight", "reg_lambda")}

rows, num = [], []
for name, group_cols in GROUPS:
    num = num + group_cols
    lr_g = make_lr(num, CAT_COLS, lr_best["C"]).fit(train, train["delayed"])
    xgb_g = fit_xgb(train, val, CAT_COLS + num, categories, xgb_params)
    for split, df in (("val", val), ("test", test)):
        scores = {
            "LR": lr_g.predict_proba(df)[:, 1],
            "XGBoost": xgb_g.predict_proba(to_xgb_frame(df, CAT_COLS + num, categories))[:, 1],
        }
        for model, p in scores.items():
            rows.append({"features": name, "n_features": len(num) + len(CAT_COLS),
                         "model": model, "split": split,
                         "pr_auc": average_precision_score(df["delayed"], p),
                         "roc_auc": roc_auc_score(df["delayed"], p)})

ablation = pd.DataFrame(rows)
ablation.to_csv(RESULTS_DIR / "ablation.csv", index=False)
ablation.pivot_table(index="features", columns=["model", "split"], values="pr_auc", sort=False).round(4)

model,LR,XGBoost,LR,XGBoost
split,val,val,test,test
features,,,,
schedule + calendar,0.2822,0.2880,0.4304,0.4363
+ congestion,0.3085,0.2971,0.4310,0.4239
+ long-run history,0.3073,0.2965,0.4404,0.4239
+ recent history,0.2975,0.2755,0.4610,0.4683


## Class imbalance: does weighting help?

About 25% of training flights are delayed, so the classes are imbalanced. The usual fixes are class weights or
resampling. We did not use them in the main models, and here we check that choice: same hyperparameters, same split,
once without weighting (our models) and once with
* `class_weight="balanced"` for logistic regression,
* `scale_pos_weight = n_negative / n_positive` for XGBoost.

We compare **raw** probabilities, because that is where weighting acts. SMOTE-style oversampling was not tried: it
creates synthetic flights by mixing flights from different days, which does not make sense with day-level history
features.

In [12]:

def expected_calibration_error(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    ece = 0.0
    for b in range(bins):
        mask = idx == b
        if mask.any():
            ece += mask.mean() * abs(p[mask].mean() - y[mask].mean())
    return ece


scale_pos_weight = (train["delayed"] == 0).sum() / (train["delayed"] == 1).sum()
lr_weighted = make_lr(NUM_COLS, CAT_COLS, lr_best["C"]).set_params(clf__class_weight="balanced")
lr_weighted.fit(train, train["delayed"])
xgb_weighted = fit_xgb(train, val, xgb_cols, categories, {**xgb_params, "scale_pos_weight": scale_pos_weight})

candidates = {
    "LR (unweighted)": lambda d: lr.predict_proba(d)[:, 1],
    "LR, balanced class weights": lambda d: lr_weighted.predict_proba(d)[:, 1],
    "XGBoost (unweighted)": lambda d: xgb.predict_proba(to_xgb_frame(d, xgb_cols, categories))[:, 1],
    f"XGBoost, positive weight {scale_pos_weight:.2f}":
        lambda d: xgb_weighted.predict_proba(to_xgb_frame(d, xgb_cols, categories))[:, 1],
}

rows = []
for name, predict in candidates.items():
    pv, pt = predict(val), predict(test)
    yt = test["delayed"].to_numpy()
    rows.append({"model": name,
                 "val_pr_auc": average_precision_score(val["delayed"], pv),
                 "val_roc_auc": roc_auc_score(val["delayed"], pv),
                 "test_pr_auc": average_precision_score(yt, pt),
                 "test_roc_auc": roc_auc_score(yt, pt),
                 "test_brier": brier_score_loss(yt, pt),
                 "test_ece": expected_calibration_error(yt, pt),
                 "test_mean_pred": pt.mean()})

imbalance = pd.DataFrame(rows)
imbalance.to_csv(RESULTS_DIR / "imbalance.csv", index=False)
print(f"December observed delay rate: {test['delayed'].mean():.3f}")
imbalance.round(4)


December observed delay rate: 0.335


,model,val_pr_auc,val_roc_auc,test_pr_auc,test_roc_auc,test_brier,test_ece,test_mean_pred
0,LR (unweighted),0.2975,0.6645,0.4610,0.6401,0.2159,0.0640,0.2953
1,"LR, balanced class weights",0.2964,0.6642,0.4600,0.6414,0.2505,0.1819,0.5169
2,XGBoost (unweighted),0.2755,0.6501,0.4683,0.6452,0.2163,0.0734,0.2689
3,"XGBoost, positive weight 3.00",0.2732,0.6494,0.4663,0.6467,0.2364,0.1527,0.4882


Weighting leaves the ranking metrics (PR-AUC, ROC-AUC) practically unchanged, but it pushes every probability up:
the mean predicted risk ends up far above the real delay rate and the calibration error more than doubles. Platt
scaling could pull the weighted probabilities back down, but then weighting would add nothing over the unweighted model.
Since calibrated probabilities are a required deliverable, the main models stay unweighted; the imbalance is handled
through the choice of metric (PR-AUC) and the precision-based decision threshold.

In [13]:

lines = [r"\begin{tabular}{lccccc}", r"\toprule",
         r"Model & Val PR-AUC & Test PR-AUC & Test ROC-AUC & Test ECE & Mean $\hat p$ \\", r"\midrule"]
for _, r in imbalance.iterrows():
    lines.append(f"{r['model']} & {r['val_pr_auc']:.3f} & {r['test_pr_auc']:.3f} & {r['test_roc_auc']:.3f} & "
                 f"{r['test_ece']:.3f} & {r['test_mean_pred']:.3f} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
(TABLE_DIR / "imbalance.tex").write_text("\n".join(lines) + "\n")


412